# 04 — Linear Regression: Predicting Audiobook Price
**Project:** Audible Audiobook Analytics  
**Team:** Team Mars  
**Author:** Bishesh Chapagain 
**Week:** 5  

This notebook builds and validates the linear regression model.  
Input: `../data/model_ready.csv`  
Output: `../models/price_model.pkl`

### Steps
1. Load data & define features
2. Train/test split
3. Scale features
4. Fit the model (statsmodels — for p-values)
5. Interpret coefficients & test hypotheses
6. Diagnostics: residuals, Q-Q plot, VIF
7. Evaluate on test set (sklearn)
8. Save model

---
## 1. Load Data & Define Features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style='whitegrid')
os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs/plots', exist_ok=True)

df = pd.read_csv('../data/model_ready.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Define feature set and target
lang_cols = [c for c in df.columns if c.startswith('lang_')]

features = (
    ['duration_minutes', 'star_score', 'log_num_ratings',
     'release_year', 'is_top_narrator', 'is_top_author']
    + lang_cols
)
target = 'price'

X = df[features]
y = df[target]

print(f'Features: {features}')
print(f'X shape: {X.shape} | y shape: {y.shape}')
print(f'\ny (price) summary:')
print(y.describe().round(2))

---
## 2. Train / Test Split

80% of rows are used to train the model, 20% are held back to test it.  
`random_state=42` ensures the split is the same every time the notebook is run.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set:  {X_train.shape[0]} rows')
print(f'Test set:      {X_test.shape[0]} rows')

---
## 3. Scale Features

Standardisation puts all features on the same scale (mean=0, std=1).  
This ensures features with large raw values (e.g. `duration_minutes`) don't  
dominate features with small values (e.g. `is_top_narrator`).  
**Important:** Fit the scaler on training data only — never on test data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

# Convert back to DataFrames so column names are preserved
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=features)

print('Scaling complete.')
X_train_scaled.describe().round(2)

---
## 4. Fit the Model with Statsmodels

We use `statsmodels` here (not sklearn) because it gives us the full  
statistical output — p-values, confidence intervals, F-statistic — which  
we need to test our hypotheses formally.

In [ ]:
# Add constant (intercept term) — required by statsmodels
X_train_sm = sm.add_constant(X_train_scaled)

ols_model = sm.OLS(y_train.values, X_train_sm).fit()

print(ols_model.summary())

---
## 5. Interpret Coefficients & Test Hypotheses

In [ ]:
# Extract coefficients and p-values into a clean table
coef_df = pd.DataFrame({
    'Feature':     ['const'] + features,
    'Coefficient': ols_model.params.values,
    'Std Error':   ols_model.bse.values,
    'p-value':     ols_model.pvalues.values,
    'Significant': ols_model.pvalues.values < 0.05
})

coef_df = coef_df[coef_df['Feature'] != 'const'].sort_values('p-value')
coef_df.round(4)

In [ ]:
# Coefficient plot — shows direction and magnitude of each feature's effect
coef_plot = coef_df.set_index('Feature')['Coefficient'].sort_values()

colors = ['steelblue' if v >= 0 else 'salmon' for v in coef_plot.values]

fig, ax = plt.subplots(figsize=(10, 6))
coef_plot.plot(kind='barh', color=colors, ax=ax)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Linear Regression Coefficients\n(Standardised — magnitude = relative importance)')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('../outputs/plots/linear_coefficients.png', dpi=150)
plt.show()

In [ ]:
# Hypothesis test summary
print('=== Hypothesis Test Results (α = 0.05) ===')
print(f'{"Feature":<22} {"p-value":>10} {"Decision"}')
print('-' * 55)
for _, row in coef_df.iterrows():
    decision = 'REJECT H₀ ✓' if row['Significant'] else 'Fail to reject H₀'
    print(f"{row['Feature']:<22} {row['p-value']:>10.4f}   {decision}")

print(f'\nOverall F-statistic p-value: {ols_model.f_pvalue:.6f}')
if ols_model.f_pvalue < 0.05:
    print('→ REJECT overall H₀: the model as a whole is statistically significant.')
else:
    print('→ Fail to reject overall H₀.')

In [ ]:
# R-squared values
print(f'R²:          {ols_model.rsquared:.4f}')
print(f'Adjusted R²: {ols_model.rsquared_adj:.4f}')
print(f'\nInterpretation: The model explains {ols_model.rsquared*100:.1f}% of the variance in audiobook price.')

---
## 6. Diagnostics

Linear regression has four key assumptions. We check each one.

In [ ]:
# Get predictions and residuals on training set
X_train_sm = sm.add_constant(X_train_scaled)
y_train_pred = ols_model.predict(X_train_sm)
residuals = y_train.values - y_train_pred

In [ ]:
# --- Assumption 1: Linearity ---
# Residuals vs Fitted — should look like a random horizontal band around 0
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(y_train_pred, residuals, alpha=0.2, color='steelblue', s=10)
ax.axhline(0, color='red', linewidth=1.2, linestyle='--')
ax.set_title('Residuals vs Fitted Values\n(Linearity & Homoscedasticity Check)')
ax.set_xlabel('Fitted Values (Predicted Price)')
ax.set_ylabel('Residuals')
plt.tight_layout()
plt.savefig('../outputs/plots/residuals_vs_fitted.png', dpi=150)
plt.show()

**What to look for:** Residuals should be randomly scattered around the 0 line with no visible pattern.  
- A funnel shape (wider spread at higher values) = **heteroscedasticity** — violates the equal variance assumption.  
- A curved pattern = **non-linearity** — the relationship isn't well captured by a straight line.

**Write your observation here:** *(e.g. 'The residuals appear randomly scattered with no strong pattern, suggesting the linearity assumption is reasonably met.')*

In [ ]:
# --- Assumption 2: Normality of Residuals ---
# Q-Q Plot — points should follow the diagonal line
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of residuals
sns.histplot(residuals, bins=60, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Residuals Distribution')
axes[0].set_xlabel('Residual')

# Q-Q plot
sm.qqplot(residuals, line='s', ax=axes[1], alpha=0.3)
axes[1].set_title('Q-Q Plot of Residuals\n(Normality Check)')

plt.tight_layout()
plt.savefig('../outputs/plots/residuals_normality.png', dpi=150)
plt.show()

**What to look for:** In the Q-Q plot, points should fall closely along the diagonal line.  
Deviation at the tails (S-shape or curve at ends) indicates the residuals are not perfectly normal.  
With ~87,000 rows, mild deviation is acceptable — the Central Limit Theorem means large samples are more robust to this.

**Write your observation here:** *(e.g. 'Residuals are approximately normally distributed. The Q-Q plot shows slight deviation at the upper tail, consistent with some remaining price outliers.')*

In [ ]:
# --- Assumption 3: No Multicollinearity ---
# VIF (Variance Inflation Factor)
# VIF > 10 = serious multicollinearity problem
# VIF 5-10 = moderate concern
# VIF < 5  = acceptable

vif_data = pd.DataFrame()
vif_data['Feature'] = features
vif_data['VIF'] = [
    variance_inflation_factor(X_train_scaled.values, i)
    for i in range(X_train_scaled.shape[1])
]
vif_data = vif_data.sort_values('VIF', ascending=False)

print('=== Variance Inflation Factors ===')
print(vif_data.round(2).to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 5]
if len(high_vif) > 0:
    print(f'\n⚠️  Features with VIF > 5 (potential multicollinearity):')
    print(high_vif.to_string(index=False))
else:
    print('\n✓ All VIF values below 5 — no multicollinearity concern.')

In [ ]:
# VIF bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['salmon' if v > 5 else 'steelblue' for v in vif_data['VIF']]
sns.barplot(data=vif_data, x='VIF', y='Feature', palette=colors, ax=ax)
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.2, label='VIF = 5')
ax.axvline(10, color='red',    linestyle='--', linewidth=1.2, label='VIF = 10')
ax.set_title('Variance Inflation Factor per Feature')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/vif_scores.png', dpi=150)
plt.show()

**Write your observation here:** *(e.g. 'All VIF values are below 5, confirming no significant multicollinearity between features. The model's coefficients are reliable.')*

---
## 7. Evaluate on Test Set

In [ ]:
# Train sklearn model on the same split (for predictions and saving)
sk_model = LinearRegression()
sk_model.fit(X_train_scaled, y_train)

y_pred = sk_model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('=== Test Set Performance ===')
print(f'R²   (R-squared):               {r2:.4f}')
print(f'RMSE (Root Mean Squared Error): {rmse:.4f}')
print(f'MAE  (Mean Absolute Error):     {mae:.4f}')
print(f'\nInterpretation: On average, predictions are off by ${mae:.2f}')

In [ ]:
# Actual vs Predicted scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.2, color='steelblue', s=10)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_title('Actual vs Predicted Price (Test Set)')
ax.set_xlabel('Actual Price')
ax.set_ylabel('Predicted Price')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/actual_vs_predicted_price.png', dpi=150)
plt.show()

**What to look for:** Points should cluster tightly around the red dashed diagonal line.  
The further points scatter away from that line, the worse the predictions are.

**Write your observation here:** *(e.g. 'The model captures the general price trend but shows wider spread at higher price values, suggesting the model underperforms on premium audiobooks.')*

---
## 8. Save Model

In [ ]:
# Save both the trained model and the scaler
# The scaler must be saved too — the app needs to scale inputs the same way
joblib.dump(sk_model, '../models/price_model.pkl')
joblib.dump(scaler,   '../models/price_scaler.pkl')
joblib.dump(features, '../models/linear_features.pkl')

print('Saved:')
print('  ../models/price_model.pkl')
print('  ../models/price_scaler.pkl')
print('  ../models/linear_features.pkl')